**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Numerical Linear Algebra

The most practically valuable math course in the curriculum: why solves explode, which factorization to use when, how Krylov methods solve systems too big to factor, and the randomized SVD that makes modern data sizes tractable — every method checked against LAPACK ground truth.

## 1. Pre-requisites

[Linear Algebra](../Linear_Algebra/Linear_Algebra.ipynb) — this is its 'in floating point, at scale' sequel.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import time
rng = np.random.default_rng(0)

---
### 🕐 Session 1 of 4 — *Conditioning & Stability* (~40 min)
**Goal:** separate the problem's sensitivity from the algorithm's sins; know when to distrust a solve.
**Builds on:** [Linear Algebra](../Linear_Algebra/Linear_Algebra.ipynb). &nbsp; **Feeds into:** Session 2 (QR & least squares).

---

## 2. Two Different Ways to Be Wrong

💡 **Intuition.** **Conditioning** is the *problem's* fault: κ(A) = σ_max/σ_min measures how much the answer moves when the data wiggles — no algorithm can beat it. **Stability** is the *algorithm's* fault: a stable algorithm returns the exact answer to a nearby problem. The rule of thumb with teeth: you lose about $\log_{10}\kappa$ digits — solve with κ=10⁸ in float64 and only ~8 of your 16 digits survive. The [GD convergence κ](../Optimization/Optimization.ipynb) and this κ are the same number wearing two hats.

In [2]:
# watch digits die exactly on schedule
for kappa in [1e2, 1e6, 1e10, 1e14]:
    n_dim = 50
    U, _ = np.linalg.qr(rng.standard_normal((n_dim, n_dim)))
    V, _ = np.linalg.qr(rng.standard_normal((n_dim, n_dim)))
    s = np.logspace(0, -np.log10(kappa), n_dim)
    A = U @ np.diag(s) @ V.T
    x_true = rng.standard_normal(n_dim)
    x_hat = np.linalg.solve(A, A @ x_true)
    rel = np.linalg.norm(x_hat - x_true) / np.linalg.norm(x_true)
    print(f"κ = {kappa:.0e}: relative error {rel:.1e}   (predicted ~ κ·ε ≈ {kappa*2.2e-16:.1e})")

κ = 1e+02: relative error 4.3e-15   (predicted ~ κ·ε ≈ 2.2e-14)
κ = 1e+06: relative error 7.9e-12   (predicted ~ κ·ε ≈ 2.2e-10)
κ = 1e+10: relative error 4.4e-08   (predicted ~ κ·ε ≈ 2.2e-06)
κ = 1e+14: relative error 7.8e-04   (predicted ~ κ·ε ≈ 2.2e-02)


**The classic self-inflicted wound:** solving least squares via the normal equations *squares* the condition number ($\kappa(A^TA) = \kappa(A)^2$). The fix is Session 2.

---
### 🕐 Session 2 of 4 — *QR & Least Squares Done Right* (~35 min)
**Goal:** orthogonalization as the workhorse; Householder vs Gram-Schmidt; the κ² trap escaped.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (Krylov).

---

## 3. Factor, Don't Invert

💡 **Intuition.** QR rewrites $A = QR$ (orthonormal × triangular): least squares becomes $R x = Q^T b$ — a stable triangular solve at the *original* κ, not κ². And not all orthogonalizations are equal: classical Gram–Schmidt loses orthogonality catastrophically in ill-conditioned bases; **Householder reflections** (LAPACK's choice) keep $Q^TQ = I$ to machine precision. Same math, different arithmetic — the whole soul of this course.

In [3]:
def gram_schmidt(A):
    Q = A.astype(float).copy()
    for j in range(A.shape[1]):
        for i in range(j):
            Q[:, j] -= (Q[:, i] @ Q[:, j]) * Q[:, i]
        Q[:, j] /= np.linalg.norm(Q[:, j])
    return Q

# a genuinely treacherous basis: 40 nearly-IDENTICAL columns
n_dim = 40
u = rng.standard_normal(n_dim)
A = u[:, None] + 1e-7 * rng.standard_normal((n_dim, n_dim))
Q_gs = gram_schmidt(A)
Q_hh, _ = np.linalg.qr(A)                            # Householder under the hood
print(f"‖QᵀQ − I‖  Gram-Schmidt: {np.abs(Q_gs.T @ Q_gs - np.eye(n_dim)).max():.1e}")
print(f"‖QᵀQ − I‖  Householder:  {np.abs(Q_hh.T @ Q_hh - np.eye(n_dim)).max():.1e}")

‖QᵀQ − I‖  Gram-Schmidt: 5.9e-08
‖QᵀQ − I‖  Householder:  6.7e-16


In [4]:
# The κ² kill shot: Läuchli's matrix (κ ≈ 1.4e8, comfortably solvable — at κ, not κ²)
eps = 1e-8                                            # < √(machine epsilon): the trap springs
A = np.array([[1.0, 1.0], [eps, 0.0], [0.0, eps]])
x_ref = np.array([1.0, 1.0])                          # KNOWN solution (the oracle)
b = A @ x_ref

print(f"cond(A)    = {np.linalg.cond(A):.1e}   ← fine for float64")
print(f"cond(AᵀA)  = {np.linalg.cond(A.T @ A):.1e}   ← κ² rounded AᵀA to exactly singular!")
try:
    x_normal = np.linalg.solve(A.T @ A, A.T @ b)
    print("normal equations:", x_normal)
except np.linalg.LinAlgError as e:
    print(f"normal equations: LinAlgError ({e}) — total failure")
Q, R = np.linalg.qr(A)
x_qr = np.linalg.solve(R, Q.T @ b)
print(f"QR:               {x_qr}   (exact — error {np.abs(x_qr - x_ref).max():.1e})")

cond(A)    = 1.4e+08   ← fine for float64
cond(AᵀA)  = inf   ← κ² rounded AᵀA to exactly singular!
normal equations: LinAlgError (Singular matrix) — total failure
QR:               [1. 1.]   (exact — error 2.2e-16)


---
### 🕐 Session 3 of 4 — *Krylov Methods: Conjugate Gradients* (~40 min)
**Goal:** solve systems you can only MULTIPLY by; watch κ set the convergence rate.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (randomized SVD).

---

## 4. Solving Without Factoring

💡 **Intuition.** When $A$ is huge and sparse (a [graph Laplacian](../../Intro_DSP/Graph_Signal_Processing.ipynb)!, a discretized PDE), you can afford $Av$ products but never a factorization. **Krylov methods** build the best answer inside $\mathrm{span}\{b, Ab, A^2b, \dots\}$ — the subspace that matrix-multiplies give you for free. **CG** (for SPD $A$) is its masterpiece: each iteration one product, error contracting like $(\frac{\sqrt\kappa - 1}{\sqrt\kappa + 1})^k$ — *√κ, not κ* ([momentum's](../../Intro_Mach_Learn/Training_Dynamics.ipynb) secret sibling). Preconditioning = warping the problem to shrink κ before you start.

In [5]:
def cg(Amul, b, iters):
    x = np.zeros_like(b); r = b.copy(); p = r.copy()
    errs = [np.linalg.norm(r)]
    for _ in range(iters):
        Ap = Amul(p)
        alpha = (r @ r) / (p @ Ap)
        x += alpha * p
        r_new = r - alpha * Ap
        beta = (r_new @ r_new) / (r @ r)
        p = r_new + beta * p; r = r_new
        errs.append(np.linalg.norm(r))
    return x, np.array(errs)

n_dim = 400
plt.figure(figsize=(8, 3))
for kappa in [1e2, 1e4]:
    Qm, _ = np.linalg.qr(rng.standard_normal((n_dim, n_dim)))
    s = np.logspace(0, -np.log10(kappa), n_dim)
    A = Qm @ np.diag(s) @ Qm.T                      # SPD with known κ
    b = rng.standard_normal(n_dim)
    x_cg, errs = cg(lambda v: A @ v, b, 250)
    rate = (np.sqrt(kappa)-1)/(np.sqrt(kappa)+1)
    plt.semilogy(errs/errs[0], label=f"κ={kappa:.0e} (theory rate {rate:.3f}/iter)")
    # ORACLE: CG's answer vs direct solve
    print(f"κ={kappa:.0e}: ‖x_CG − x_direct‖/‖x‖ = "
          f"{np.linalg.norm(x_cg - np.linalg.solve(A, b))/np.linalg.norm(x_cg):.1e}")
plt.legend(); plt.xlabel("iteration"); plt.ylabel("relative residual")
plt.title("CG: √κ convergence — and never a factorization in sight")
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

κ=1e+02: ‖x_CG − x_direct‖/‖x‖ = 2.7e-15
κ=1e+04: ‖x_CG − x_direct‖/‖x‖ = 2.9e-03


/tmp/ipykernel_2977446/3278043996.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()


---
### 🕐 Session 4 of 4 — *Randomized SVD* (~35 min)
**Goal:** sketch first, factor later: near-best low-rank approximations at a fraction of the cost.
**Builds on:** Session 3; [Linear Algebra](../Linear_Algebra/Linear_Algebra.ipynb) S4.

---

## 5. The Randomized Revolution

💡 **Intuition.** To find a rank-$k$ approximation you don't need all of $A$ — you need its **range**. Hit $A$ with $k{+}p$ random vectors: with overwhelming probability ([concentration](../Concentration/Concentration_Inequalities.ipynb)!) the sketch $Y = A\Omega$ spans nearly the same top subspace. Orthonormalize the sketch, project, SVD the small core: $O(mnk)$ instead of $O(mn\min(m,n))$, and accuracy within a hair of Eckart–Young optimal.

In [6]:
def rsvd(A, k, p=8, power=1):
    Omega = rng.standard_normal((A.shape[1], k + p))
    Y = A @ Omega                                     # capture the range from a sketch
    for _ in range(power):                            # one power iteration sharpens the subspace
        Y = A @ (A.T @ Y)
    Q, _ = np.linalg.qr(Y)
    B = Q.T @ A                                       # small (k+p) × n core
    Ub, s, Vt = np.linalg.svd(B, full_matrices=False)
    return Q @ Ub[:, :k], s[:k], Vt[:k]

# a big low-rank-plus-noise matrix
m, n_c, k = 1500, 1200, 20
A = rng.standard_normal((m, k)) @ rng.standard_normal((k, n_c)) + 0.05*rng.standard_normal((m, n_c))

tic = time.perf_counter(); U_f, s_f, Vt_f = np.linalg.svd(A, full_matrices=False); t_full = time.perf_counter()-tic
tic = time.perf_counter(); U_r, s_r, Vt_r = rsvd(A, k); t_rand = time.perf_counter()-tic

# ORACLE: singular values must match the full SVD
print("max relative error in top-20 singular values:", np.abs(s_r - s_f[:k]).max()/s_f[0])
err_opt  = np.linalg.norm(A - U_f[:, :k]*s_f[:k] @ Vt_f[:k])
err_rand = np.linalg.norm(A - U_r*s_r @ Vt_r)
print(f"‖A − Â_k‖: optimal {err_opt:.1f}   randomized {err_rand:.1f}   (ratio {err_rand/err_opt:.4f})")
print(f"time: full SVD {t_full*1e3:.0f} ms   randomized {t_rand*1e3:.0f} ms   ({t_full/t_rand:.0f}x)")

max relative error in top-20 singular values: 9.900614752923446e-15
‖A − Â_k‖: optimal 66.0   randomized 66.0   (ratio 1.0000)
time: full SVD 252 ms   randomized 5 ms   (55x)


## 6. Conclusion

κ prices every digit; QR dodges the κ² trap Gram–Schmidt falls into; CG solves at √κ per iteration with only matrix-multiplies; random sketches capture ranges with probability on your side. This is the difference between knowing linear algebra and *computing* it.

---
## Where next

- [Random Matrix Theory](../Random_Matrix_Theory/Random_Matrix_Theory.ipynb) — why those random sketches work so well.
- [Graph Signal Processing](../../Intro_DSP/Graph_Signal_Processing.ipynb) — Krylov's natural habitat.
- [Scaling Neural Networks](../../Intro_Mach_Learn/Scale_NN/Scale_NN.ipynb) — the same flop-counting instincts, applied to training.